# Skills Pattern (스킬 기반 패턴 - Progressive Disclosure)

**Progressive Disclosure**는 에이전트가 필요한 정보를 **필요할 때만** 로드하는 방식입니다.
시스템 프롬프트에는 스킬의 **간단한 설명**만 노출하고, 상세 스키마·비즈니스 로직은 `load_skill` 도구로 불러옵니다.

**참고**: [LangChain 공식 문서 - Build a SQL assistant with on-demand skills](https://docs.langchain.com/oss/python/langchain/multi-agent/skills-sql-assistant)

## 1. 스킬 정의 (Skill TypedDict)

각 스킬은 name, description(시스템 프롬프트에 노출), content(도구 호출 시 로드)를 가집니다.

In [ ]:
# Skill 데이터 구조 정의
class Skill(TypedDict):
# 에이전트가 사용할 수 있는 모든 스킬 목록
    # ------------------------------------------------------------
    # 매출 분석 스킬
    # ------------------------------------------------------------
        # 에이전트에게 노출되는 "스킬 설명"
        # → SkillMiddleware에서 system prompt에 삽입됨
        # 실제 스킬 내용
        # → load_skill tool 호출 시 모델 컨텍스트에 삽입됨
## 테이블
## 비즈니스 로직
## 예시 쿼리
    # ------------------------------------------------------------
    # 재고 관리 스킬
    # ------------------------------------------------------------
        # 재고 도메인 관련 메타 설명 → 모델이 "재고 관련 질문인지" 판단할 때 사용
        # 실제 재고 관리 스킬 내용
## 테이블
## 비즈니스 로직

## 2. load_skill 도구

스킬 이름으로 전체 content를 로드해 에이전트 컨텍스트에 넣습니다.

In [ ]:
def load_skill(skill_name: str) -> str:
    # ------------------------------------------------------------
    # SKILLS 레지스트리에서 요청된 스킬 검색
    # ------------------------------------------------------------
        # 스킬 이름이 일치하면 해당 스킬 반환
            # 반환된 문자열은 ToolMessage로 변환되어 모델 컨텍스트에 삽입됨
    # 사용자 또는 에이전트가 잘못된 스킬 이름을 요청했을 때 사용 가능한 스킬 목록을 안내

## 3. SkillMiddleware

시스템 프롬프트에 **스킬 설명만** 추가하고, load_skill 도구를 등록합니다.

In [ ]:
class SkillMiddleware(AgentMiddleware):
    # Middleware에서 사용할 Tool 등록
    def __init__(self):
        # 각 스킬을 Markdown 리스트 형태로 구성
        # 문자열로 합쳐 저장
    def wrap_model_call(
        # 스킬 안내 텍스트 구성
        # 현재 시스템 프롬프트 가져오기
        # system_prompt가 없으면 system_message에서 추출
        # 둘 다 없으면 빈 문자열 사용
        # 기존 시스템 프롬프트 + 스킬 목록 결합
        # 수정된 시스템 프롬프트로 모델 호출 진행
        # override()는 ModelRequest를 복사하면서 일부 필드만 수정

## 4. 에이전트 생성

SkillMiddleware와 checkpointer로 대화 상태를 유지합니다.

## 5. Progressive Disclosure 테스트

질문 시 에이전트가 load_skill(sales_analytics)을 호출한 뒤 스키마를 활용해 쿼리를 작성합니다.

In [ ]:
# 대화 스레드 ID 생성
# Agent 실행 설정(config)
        # 대화 메시지 리스트
        # HumanMessage는 사용자 입력을 의미
    # pretty_print가 있으면 보기 좋은 출력 형식 사용

## 주요 포인트 정리

1. **Progressive Disclosure**: 스킬 설명만 먼저 노출, 상세 내용은 load_skill로 필요 시 로드
2. **SkillMiddleware**: 시스템 프롬프트에 스킬 설명 추가, load_skill 도구 등록
3. **컨텍스트 절약**: 필요한 1~2개 스킬만 로드해 토큰 사용 최소화
4. **팀 확장성**: 도메인별 스킬을 독립적으로 추가·유지보수 가능

**다음 단계**:
- [350_Human_In_The_Loop.py](350_Human_In_The_Loop.py)에서 Human-in-the-Loop 패턴 학습
- [310_Subagents_Pattern.py](310_Subagents_Pattern.py)에서 다른 멀티 에이전트 패턴 비교